In [218]:
# necessary libraries 

import pandas as pd
import numpy as np
import string
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV

In [219]:
#loading dataset
df = pd.read_csv('data_balanced.csv')

In [220]:
df.shape

(203, 47)

In [221]:
# preprocessing the texts: lowercasing and removing punctuation 

def preproc(sentence):
    sentence = sentence.lower()
    sentence = ''.join([i for i in sentence if i not in string.punctuation])
    return sentence

In [222]:
df['Ptext'] = df['text'].apply(preproc) #adding preprocessed texts to the dataset

In [226]:
subset = df[df["label"].isin([4,5])] #subsetting dataset into a dataset with only 2 specific labels

In [227]:
subset = subset.sample(frac=1) #shuffling the dataset

In [228]:
subset.head()

,ID,text,read_score,label,Time.Point,N_switches,T_write_ms,T_plan_ms,N_char_write,N_char_plan_intro,...,x0034,x0014,x0024,x0013,x0123,x0124,x0134,x0234,x1234,Ptext
184,6459816,in der schule hat unsere klasse eine debatte ü...,46,5,pre,13.0,2529615.0,1597633.0,931.0,26.0,...,True,False,False,False,False,False,False,False,False,in der schule hat unsere klasse eine debatte ü...
131,6019072,Sollte man ab den. jahrgang Theaterunterricht ...,59,4,post,27.0,1649630.0,390193.0,1574.0,0.0,...,False,False,False,False,False,False,False,False,False,sollte man ab den jahrgang theaterunterricht b...
179,4968717,Die schülervertreter unserer schulehaben vorge...,32,5,pre,10.0,623016.0,1281658.0,612.0,221.0,...,False,True,False,True,True,True,True,False,True,die schülervertreter unserer schulehaben vorge...
175,6430699,Freiwillige feuerwehrEs ist ehrenamtlichMann b...,24,5,pre,5.0,256854.0,180741.0,138.0,22.0,...,True,True,True,False,False,True,True,True,True,freiwillige feuerwehres ist ehrenamtlichmann b...
152,6367022,Sollte man ein Ehrenamt ausüben? - In meiner K...,59,4,fup,5.0,1602919.0,83692.0,1923.0,0.0,...,True,True,True,False,False,True,True,True,True,sollte man ein ehrenamt ausüben in meiner kla...


In [229]:
subset.shape

(83, 48)

### Tf-idf representation + 1) SVM ; 2) Logistic Regression

In [230]:
X_train, X_test, y_train, y_test = train_test_split(subset[['Ptext','read_score', 'MW_B001', 'MW_C001', 'MW_D001', 'MW_D002', 'MW_E211', 'MW_E221', 'MW_E231', 'MW_E102', 'MW_E101', 'MW_ZS', 'MW_SYS', 'MW_PRA']], subset['label'], test_size=0.3, random_state=42)

In [231]:
tfidf = TfidfVectorizer()

train_tfidf =  tfidf.fit_transform(X_train['Ptext'])
test_tfidf = tfidf.transform(X_test['Ptext'])

In [232]:
train_tfidf

<58x2785 sparse matrix of type '<class 'numpy.float64'>'
	with 7183 stored elements in Compressed Sparse Row format>

In [233]:
np.unique(y_train)

array([4, 5], dtype=int64)

In [234]:
# support vector machine 

svm = SVC(kernel='linear', C=10, random_state=42)
svm.fit(train_tfidf, y_train)

SVC(C=10, kernel='linear', random_state=42)

In [235]:
y_pred_test_svm = svm.predict(test_tfidf)

In [236]:
#classification report on test set SVM
print(classification_report(y_test, y_pred_test_svm, digits=3))

              precision    recall  f1-score   support

           4      0.529     0.900     0.667        10
           5      0.875     0.467     0.609        15

    accuracy                          0.640        25
   macro avg      0.702     0.683     0.638        25
weighted avg      0.737     0.640     0.632        25



In [237]:
# logistic regression

lr = LogisticRegression(random_state=42)
lr.fit(train_tfidf, y_train)

y_pred_test_lr = lr.predict(test_tfidf)

In [238]:
#classification report on test set Log Reg
print(classification_report(y_test, y_pred_test_lr, digits=3))

              precision    recall  f1-score   support

           4      0.474     0.900     0.621        10
           5      0.833     0.333     0.476        15

    accuracy                          0.560        25
   macro avg      0.654     0.617     0.548        25
weighted avg      0.689     0.560     0.534        25

